In [3]:
import numpy as np
from scipy import stats

# ==============================================================================
# ПАРАМЕТРИ ВАРІАНТА 6
# Місто: Чернігів
# Липнева середня температура за Практикою 1: 20 °C
# ==============================================================================
VARIANT = 6
CITY = "Чернігів"
JULY_BASE_TEMP = 20  # Табличне липневе значення для 6 варіанта

print(f"=== ПРАКТИКА 12: Варіант {VARIANT} ({CITY}) ===")

# ------------------------------------------------------------------------------
# ЗАВДАННЯ 1. Генерація вибірки денних температур
# ------------------------------------------------------------------------------
np.random.seed(VARIANT)  # Фіксуємо seed номером варіанта
n = 30
daily_temps = np.random.normal(loc=JULY_BASE_TEMP, scale=2.5, size=n)

sample_mean = daily_temps.mean()
sample_std = daily_temps.std(ddof=1)  # ddof=1 для вибіркового стандартного відхилення s

print("\n=== Завдання 1: Згенерована вибірка ===")
print("Масив температур (°C):")
print(np.round(daily_temps, 2))
print(f"Обсяг вибірки (n): {n}")
print(f"Вибіркове середнє (x̄): {sample_mean:.4f} °C")
print(f"Вибіркове ст. відхилення (s, ddof=1): {sample_std:.4f} °C")

# ------------------------------------------------------------------------------
# ЗАВДАННЯ 2. Довірчий інтервал для середньої температури
# ------------------------------------------------------------------------------
se_mean = sample_std / np.sqrt(n)

# 95% довірчий інтервал через t-розподіл Стьюдента (df = n - 1)
ci_mean_95 = stats.t.interval(0.95, df=n - 1, loc=sample_mean, scale=se_mean)

print("\n=== Завдання 2: 95% Довірчий інтервал для середнього (t-розподіл) ===")
print(f"Стандартна похибка (SE): {se_mean:.4f} °C")
print(f"95% ДІ для μ: ({ci_mean_95[0]:.4f}, {ci_mean_95[1]:.4f}) °C")

explanation_task2 = """
[Аналіз до Завдання 2]
Чому не можна використовувати z-критичне значення:
Справжнє генеральне стандартне відхилення σ кліматичної популяції є НЕВІДОМИМ.
Ми використовуємо його вибіркову оцінку s, що додає додаткову невизначеність.
Оскільки обсяг вибірки малий (n = 30), невизначеність оцінки s є суттєвою.
Відповідно до теорії з Лекції 13, за невідомого σ та малого n застосування z-статистики призведе до
невиправданого звуження інтервалу та зниження реального рівня довіри нижче 95%.
Тому необхідно використовувати t-розподіл Стьюдента з ступенями свободи df = n - 1 = 29.
"""
print(explanation_task2)

# ------------------------------------------------------------------------------
# ЗАВДАННЯ 3. Довірчий інтервал для частки
# ------------------------------------------------------------------------------
above_july_base = daily_temps > JULY_BASE_TEMP
p_hat = above_july_base.mean()

n_p = n * p_hat
n_1p = n * (1 - p_hat)

print("=== Завдання 3: Довірчий інтервал для частки p_hat ===")
print(f"Частка днів з температурою > {JULY_BASE_TEMP} °C (p_hat): {p_hat:.4f} ({p_hat * 100:.1f}%)")
print(f"Перевірка умови нормального наближення: n * p_hat = {n_p:.1f}, n * (1 - p_hat) = {n_1p:.1f}")

# Обчислення ДІ для частки
se_p = np.sqrt(p_hat * (1 - p_hat) / n)
ci_prop_95 = stats.norm.interval(0.95, loc=p_hat, scale=se_p)

print(f"Стандартна похибка частки (SE_p): {se_p:.4f}")
print(f"95% ДІ для частки p: ({ci_prop_95[0]:.4f}, {ci_prop_95[1]:.4f})")

contains_half = ci_prop_95[0] <= 0.5 <= ci_prop_95[1]
print(f"Чи покриває 95% ДІ теоретичне значення p = 0.5? {'ТАК' if contains_half else 'НІ'}")

explanation_task3 = f"""
[Аналіз до Завдання 3]
1. Перевірка умови: Обидва значення (n*p_hat = {n_p:.1f} та n*(1-p_hat) = {n_1p:.1f}) є більше або дорівнюють 5
   (n*p_hat ≥ 5 та n*(1-p_hat) ≥ 5), отже умову нормального наближення виконано.
2. Покриття p = 0.5: Інтервал ({ci_prop_95[0]:.4f}, {ci_prop_95[1]:.4f}) {'накриває' if contains_half else 'не накриває'} теоретичне значення 0.5.
   Якщо б одна конкретна вибірка дала ДІ, який не накрив 0.5, це не означало б помилку в розрахунках.
   Рівень довіри 95% означає, що при багаторазовому повторенні дослідження приблизно 5% усіх побудованих
   інтервалів через випадкові вибіркові коливання НЕ накриватимуть істинне значення.
"""
print(explanation_task3)

# ------------------------------------------------------------------------------
# ЗАВДАННЯ 4. Ширина інтервалу при різних рівнях довіри
# ------------------------------------------------------------------------------
levels = [0.90, 0.95, 0.99]
ci_results = []

print("=== Завдання 4: Ширина ДІ при різних рівнях довіри ===")
print(f"{'Рівень довіри (1-α)':<20} | {'Нижня межа':<12} | {'Верхня межа':<12} | {'Ширина інтервалу':<18}")
print("-" * 70)

for lvl in levels:
    ci = stats.t.interval(lvl, df=n - 1, loc=sample_mean, scale=se_mean)
    width = ci[1] - ci[0]
    ci_results.append((lvl, ci[0], ci[1], width))
    print(f"{lvl * 100:.0f}% {'':<16} | {ci[0]:<12.4f} | {ci[1]:<12.4f} | {width:<18.4f}")

explanation_task4 = """
[Аналіз до Завдання 4]
Чому ширина зростає зі збільшенням рівня довіри:
Щоб бути більш впевненими (наприклад, 99% замість 90%), що побудований інтервал покриє істинне значення μ,
ми змушені розширювати інтервал (захоплювати більшу критичну область t_{crit}).
Що треба «заплатити» за вужчий інтервал при збереженні рівня довіри 99%:
Заплатити доведеться збільшенням обсягу вибірки (n). Оскільки ширина інтервалу обернено пропорційна √n,
збільшення n дозволить зменшити SE і зробити 99% інтервал вужчим без втрати надійності.
"""
print(explanation_task4)

# ------------------------------------------------------------------------------
# ЗАВДАННЯ 5. Інтерпретація результату словами
# ------------------------------------------------------------------------------
print("=== Завдання 5: Коректна частотна інтерпретація ===")
explanation_task5 = f"""
Побудований 95% довірчий інтервал ({ci_mean_95[0]:.2f} °C, {ci_mean_95[1]:.2f} °C) отримано за методикою,
яка гарантує точність у 95% випадків. Якби ми згенерували 100 незалежних вибірок однакового обсягу (n = 30)
і для кожної побудували 95% довірчий інтервал за цією ж формулою, то приблизно 95 із цих 100 інтервалів
містили б у собі істинну середньомісячну температуру липневих днів, а близько 5 інтервалів не накрили б її.
Конкретний обчислений інтервал є результатом однієї з таких процедур.
"""
print(explanation_task5)

=== ПРАКТИКА 12: Варіант 6 (Чернігів) ===

=== Завдання 1: Згенерована вибірка ===
Масив температур (°C):
[19.22 21.82 20.54 17.75 13.78 22.28 22.82 16.21 24.1  18.93 26.58 21.5
 19.16 23.09 20.28 20.32 20.19 19.61 21.59 22.03 20.89 24.53 16.61 18.84
 22.06 17.06 23.91 21.78 19.55 21.34]
Обсяг вибірки (n): 30
Вибіркове середнє (x̄): 20.6127 °C
Вибіркове ст. відхилення (s, ddof=1): 2.7065 °C

=== Завдання 2: 95% Довірчий інтервал для середнього (t-розподіл) ===
Стандартна похибка (SE): 0.4941 °C
95% ДІ для μ: (19.6020, 21.6233) °C

[Аналіз до Завдання 2]
Чому не можна використовувати z-критичне значення:
Справжнє генеральне стандартне відхилення σ кліматичної популяції є НЕВІДОМИМ.
Ми використовуємо його вибіркову оцінку s, що додає додаткову невизначеність.
Оскільки обсяг вибірки малий (n = 30), невизначеність оцінки s є суттєвою.
Відповідно до теорії з Лекції 13, за невідомого σ та малого n застосування z-статистики призведе до
невиправданого звуження інтервалу та зниження реального р

# Відповіді на контрольні питання
**Практична робота №12: Довірчі інтервали для середнього і частки**
**Студент:** Варіант 6 (м. Чернігів)

---

### 1. Яка різниця між точковою оцінкою і довірчим інтервалом — чому саме число (наприклад, $\bar{x}$) недостатньо для повної відповіді?

* **Точкова оцінка (Point Estimate):** Це одне конкретне значення (наприклад, вибіркове середнє $\bar{x} = 20.29 \text{ °C}$), обчислене за даними вибірки, яке слугує найкращим припущенням щодо значення генерального параметра ($\mu$).
* **Довірчий інтервал (Confidence Interval):** Це діапазон значень $[L, U]$, побудований навколо точкової оцінки з урахуванням вибіркової невизначеності (стандартної похибки).

**Чому точкової оцінки недостатньо:**
1. **Відсутність міри точності:** Точкова оцінка не дає жодної інформації про те, наскільки вона близька до істинного генерального параметра.
2. **Вплив випадковості:** Через вибіркову мінливість точкова оцінка майже ніколи не збігається з істинним параметром $\mu$ абсолютно точно ($P(\bar{x} = \mu) = 0$ для неперервних величин).
3. **Не враховується обсяг вибірки:** Оцінка $\bar{x} = 20.29$, отримана за $n = 5$ спостереженнями, має значно більшу невизначеність, ніж ця ж оцінка, отримана за $n = 1000$ спостереженнями. Довірчий інтервал враховує обсяг вибірки та варіативність даних, показуючи реальну точність вимірювання.

---

### 2. Чому для довірчого інтервалу середнього при невідомому $\sigma$ і не дуже великому $n$ використовують $t$-розподіл, а не звичайний нормальний?

Для побудови інтервалу необхідно знати стандартну похибку $SE = \frac{\sigma}{\sqrt{n}}$.

1. **Невідоме $\sigma$:** У реальних дослідженнях генеральне стандартне відхилення $\sigma$ майже завжди невідоме, тому його замінюють вибірковою оцінкою $s$.
2. **Додаткова випадковість:** Оскільки $s$ сам по собі є випадковою величиною, яка змінюється від вибірки до вибірки, величина $T = \frac{\bar{x} - \mu}{s / \sqrt{n}}$ має більший розкид, ніж стандартна нормальна величина $Z = \frac{\bar{x} - \mu}{\sigma / \sqrt{n}}$.
3. **Властивості $t$-розподілу:** $t$-розподіл Стьюдента має «важчі хвости» (heavier tails) порівняно з нормальним $Z$-розподілом. Це враховує додаткову невизначеність від оцінювання $\sigma$ через $s$. При малих $n$ ($n < 30 \dots 50$) використання $Z$-критерію призвело б до занадто вузького інтервалу, реальний рівень довіри якого був би меншим за заявлені 95%.

---

### 3. Що насправді означає "95% довіри" — сформулюйте правильно, назвавши явно, чому поширене формулювання через "ймовірність для цього інтервалу" є помилковим.

* **Коректне частотне визначення:**
  «95% довіри» — це властивість **методу (процедури)** побудови інтервалу, а не конкретного обчисленого числового діапазону. Це означає: якби ми повторили дослідження багато разів (згенерували 100 незалежних вибірок однаковості обсягу $n$) і кожного разу обчислили інтервал за цією формулою, то приблизно 95% побудованих інтервалів накрили б істинне значення генерального параметра $\mu$.

* **Чому формулювання «з ймовірністю 95% $\mu$ лежить у цьому інтервалі» є помилковим:**
  Після того, як конкретні числові межі інтервалу (наприклад, $[19.38, 21.20]$) вже обчислені, в них немає жодної випадковості:
  * Істинне параметричне значення $\mu$ є константою (хоч і невідомою).
  * Числові межі є фіксованими числами.

  Тому істинне значення $\mu$ **або лежить** у цьому конкретному інтервалі, **або не лежить**. Ймовірність цього факту дорівнює **або 1, або 0**, але аж ніяк не 0.95. Число 95% відноситься до довгострокової надійності алгоритму побудови, а не до ймовірності для вже зафіксованих чисел.

---

### 4. Які саме три фактори визначають ширину довірчого інтервалу, і як кожен з них на неї впливає?

Формула ширини довірчого інтервалу для середнього має вигляд:
$$\text{Width} = 2 \cdot t_{crit} \cdot \frac{s}{\sqrt{n}}$$

Три основні фактори:

1. **Рівень довіри ($1 - \alpha$):**
   * *Вплив:* **Пряма залежність.** Чим вищий рівень довіри (наприклад, перехід від 90% до 99%), тим більшим є критичний коефіцієнт $t_{crit}$, і відповідно **ширшим** стає інтервал.
2. **Мінливість/варіативність даних ($s$ або $\sigma$):**
   * *Вплив:* **Пряма залежність.** Чим більший розкид даних у генеральній сукупності (більше вибіркове стандартне відхилення $s$), тим більша невизначеність і тим **ширшим** робиться інтервал.
3. **Обсяг вибірки ($n$):**
   * *Вплив:* **Обернена квадратична залежність.** Обсяг вибірки знаходиться у знаменнику під знаком квадратного кореня ($\sqrt{n}$). Збільшення $n$ зменшує стандартну похибку $SE$, що робить інтервал **вужчим** (для зменшення ширини вдвічі потрібно збільшити вибірку в 4 рази).